# Тема 6. Конструирование и отбор признаков. Регрессия с регуляризацией

«Garbage in — garbage out» — одно из главных правил машинного обучения. Хорошо подготовленные признаки часто важнее выбора алгоритма. Простая модель на качественных данных нередко превосходит сложный ансамбль на «сырых».

Это занятие охватывает три взаимосвязанные темы:
- **Конструирование признаков** — как создавать полезные признаки из текстов, дат, геоданных.
- **Отбор признаков** — как избавиться от шума.
- **Ridge и Lasso регрессии** — регуляризованная линейная регрессия на практике.

### Содержание
1. [Извлечение признаков из разных типов данных](#1.-Извлечение-признаков)
2. [Преобразование признаков](#2.-Преобразование-признаков)
3. [Отбор признаков](#3.-Отбор-признаков)
4. [Ridge и Lasso регрессии на практике](#4.-Ridge-и-Lasso-регрессии)
5. [Практика](#5.-Практика)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
sns.set()
from matplotlib import pyplot as plt
from scipy.spatial.distance import euclidean
from scipy.stats import shapiro, lognorm

from sklearn.datasets import fetch_california_housing, make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import (
    SelectKBest, SelectFromModel, SequentialFeatureSelector,
    VarianceThreshold, f_classif, f_regression
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import (
    Lasso, LassoCV, LinearRegression, LogisticRegression, Ridge, RidgeCV
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures, StandardScaler

%config InlineBackend.figure_format = 'svg'

---
## 1. Извлечение признаков

В реальных задачах данные редко поступают в виде готовой матрицы. Рассмотрим основные типы данных и способы работы с ними.

### 1.1 Тексты

Текст — один из самых богатых источников признаков. Первый шаг — **токенизация**: разбиение на токены (слова, символы, n-граммы).

#### Bag of Words — «мешок слов»

Каждый документ превращается в вектор частот слов из общего словаря.

In [ ]:
texts = ["у меня есть кошка", "у тебя есть собака", "у тебя и у меня есть кошка и собака"]

# Ручная реализация для понимания
vocab = list(enumerate(sorted(set(w for s in texts for w in s.split()))))
print("Словарь:", vocab)

def vectorize(words):
    vec = np.zeros(len(vocab))
    for i, word in vocab:
        vec[i] = words.count(word)
    return vec

print("\nВекторы:")
for s in texts:
    print(f"  '{s}'")
    print(f"   -> {vectorize(s.split())}")

In [ ]:
# На практике — CountVectorizer из sklearn
vect = CountVectorizer()
X_bow = vect.fit_transform(texts).toarray()
print("Признаки:", vect.get_feature_names_out())
print("Матрица:\n", X_bow)

**Проблема Bag of Words:** порядок слов теряется. «У меня нет кошки» и «Нет, у меня есть кошка» станут похожими при одинаковых словах.

Решение — **N-граммы**: рассматривать пары, тройки слов подряд.

In [ ]:
vect_ngram = CountVectorizer(ngram_range=(1, 2))
X_ngram = vect_ngram.fit_transform(["нет у меня кошки", "у меня есть кошка"]).toarray()
print("Признаки (1- и 2-граммы):", vect_ngram.get_feature_names_out())
print("Матрица:\n", X_ngram)

In [ ]:
# Символьные N-граммы: устойчивы к опечаткам, учитывают морфологию
vect_char = CountVectorizer(ngram_range=(3, 3), analyzer='char_wb')
n1, n2, n3, n4 = vect_char.fit_transform(
    ['андерсен', 'петерсен', 'петров', 'смит']
).toarray()

print("Расстояния на основе символьных 3-грамм:")
print(f"  андерсен ↔ петерсен : {euclidean(n1, n2):.2f}  (похожие окончания)")
print(f"  петерсен ↔ петров   : {euclidean(n2, n3):.2f}")
print(f"  петров   ↔ смит     : {euclidean(n3, n4):.2f}  (очень разные)")

#### TF-IDF: взвешивание по редкости

Bag of Words одинаково считает «и» и «геофизика». **TF-IDF** увеличивает вес редких, предметно-специфичных слов:

$$\text{TF-IDF}(t, d) = \text{tf}(t, d) \cdot \log\frac{N + 1}{n_t + 1} + 1$$

где $\text{tf}(t,d)$ — частота термина $t$ в документе $d$, $N$ — общее число документов, $n_t$ — число документов с термином $t$.

In [ ]:
corpus = [
    "геофизика изучает строение земли",
    "сейсмология изучает землетрясения и строение земли",
    "машинное обучение применяется в геофизике и геологии",
    "нейронные сети и машинное обучение",
]

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

# Важность слов в первом документе
names = tfidf.get_feature_names_out()
scores = X_tfidf[0].toarray()[0]
important = [(w, s) for w, s in zip(names, scores) if s > 0]
important.sort(key=lambda x: -x[1])

print("TF-IDF веса в первом документе ('геофизика изучает строение земли'):")
for word, score in important:
    print(f"  {word:20s}: {score:.3f}")

### 1.2 Геопространственные данные

Геоданные встречаются в задачах недвижимости, транспорта, логистики. Основные приёмы:
- **Геокодирование**: адрес → координаты (через API Google Maps, Яндекс, OSM).
- **Расстояние до POI**: до метро, до центра, до ближайшего объекта.
- **Принадлежность к кластеру**: район, квартал, зона.

Для расстояний на поверхности земли — формула Хаверсина:

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi    = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Москва (Кремль) ↔ Санкт-Петербург (Дворцовая)
print(f"Москва ↔ Санкт-Петербург: {haversine_km(55.7514, 37.6178, 59.9390, 30.3157):.0f} км")

# Пример: признаки расстояния для задачи недвижимости
np.random.seed(42)
center_lat, center_lon = 55.75, 37.62  # центр Москвы
df_geo = pd.DataFrame({
    'lat': np.random.uniform(55.6, 55.9, 10),
    'lon': np.random.uniform(37.4, 37.9, 10),
    'price': np.random.randint(5, 25, 10),
})
df_geo['dist_to_center_km'] = df_geo.apply(
    lambda r: haversine_km(r.lat, r.lon, center_lat, center_lon), axis=1
)
print("\nПризнак 'расстояние до центра':")
print(df_geo[['lat', 'lon', 'dist_to_center_km', 'price']].round(2).head())

### 1.3 Дата и время

**Стандартные признаки из временной метки:**

In [ ]:
dates = pd.date_range('2024-01-01', periods=8, freq='11D 5h')
df_time = pd.DataFrame({'timestamp': dates})

df_time['year']       = df_time['timestamp'].dt.year
df_time['month']      = df_time['timestamp'].dt.month
df_time['day']        = df_time['timestamp'].dt.day
df_time['hour']       = df_time['timestamp'].dt.hour
df_time['weekday']    = df_time['timestamp'].dt.weekday   # 0=пн, 6=вс
df_time['is_weekend'] = (df_time['weekday'] >= 5).astype(int)
df_time['quarter']    = df_time['timestamp'].dt.quarter

df_time[['timestamp','month','day','hour','weekday','is_weekend']].head(5)

**Проблема цикличности.** Час 23 и час 0 числово далеки, но смыслово близки. Это ломает алгоритмы на основе расстояний (kNN, SVM, k-means).

**Решение — гармоническая кодировка** (sin/cos):

In [ ]:
def harmonic_encode(value, period):
    angle = 2 * np.pi * value / period
    return np.cos(angle), np.sin(angle)

# Проверка: 23:00 и 01:00 должны быть близко
print(f"Расстояние 23:00 ↔ 01:00 : {euclidean(harmonic_encode(23,24), harmonic_encode(1,24)):.3f}  <- близко")
print(f"Расстояние 09:00 ↔ 21:00 : {euclidean(harmonic_encode(9,24), harmonic_encode(21,24)):.3f}  <- далеко")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
hours = np.arange(24)
cos_h, sin_h = harmonic_encode(hours, 24)
axes[0].plot(hours, cos_h, label='cos(hour)', color='steelblue')
axes[0].plot(hours, sin_h, label='sin(hour)', color='orange')
axes[0].set_xlabel('Час')
axes[0].set_title('Гармонические признаки для часа суток')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
sc = axes[1].scatter(cos_h, sin_h, c=hours, cmap='twilight', s=60, edgecolors='white', zorder=3)
for h in [0, 6, 12, 18]:
    axes[1].annotate(f'{h}:00', (cos_h[h]+0.05, sin_h[h]+0.05), fontsize=9)
plt.colorbar(sc, ax=axes[1], label='Час')
axes[1].set_title('Часы на единичной окружности')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()

Аналогично кодируют: день недели (period=7), месяц (period=12), день года (period=365).

### 1.4 Взаимодействия признаков

Признаки в комбинации могут быть информативнее, чем по отдельности. Это **feature interactions** — инженерия на основе знания предметной области.

In [ ]:
# Пример: в задаче аренды жилья
# «цена за комнату» информативнее, чем цена и число комнат по отдельности
np.random.seed(42)
df_rent = pd.DataFrame({
    'price':    np.random.randint(30000, 200000, 200),
    'bedrooms': np.random.randint(1, 6, 200),
    'sqm':      np.random.randint(20, 150, 200),
})
df_rent['price_per_room'] = df_rent['price'] / df_rent['bedrooms'].clip(lower=0.5)
df_rent['price_per_sqm']  = df_rent['price'] / df_rent['sqm']

print("Созданные признаки-взаимодействия:")
print(df_rent[['price','bedrooms','sqm','price_per_room','price_per_sqm']].head())
print("\nТакие признаки модель не построит автоматически — нужны знания предметной области.")

---
## 2. Преобразование признаков

### 2.1 Масштабирование

Алгоритмы, основанные на расстояниях или градиентном спуске (kNN, SVM, линейные модели с регуляризацией, нейросети), чувствительны к масштабу. Деревья и случайные леса — нет.

| Метод | Формула | Результат |
|---|---|---|
| `StandardScaler` | $(x - \mu) / \sigma$ | среднее=0, std=1 |
| `MinMaxScaler` | $(x - x_{\min}) / (x_{\max} - x_{\min})$ | диапазон $[0, 1]$ |
| `RobustScaler` | $(x - \text{медиана}) / \text{IQR}$ | устойчив к выбросам |

In [ ]:
from sklearn.preprocessing import RobustScaler

data_raw = np.array([1, 1, 0, -1, 2, 1, 2, 3, -2, 4, 100], dtype=float).reshape(-1, 1)

results_scale = {
    'Исходные':      data_raw.ravel(),
    'Standard':      StandardScaler().fit_transform(data_raw).ravel(),
    'MinMax':        MinMaxScaler().fit_transform(data_raw).ravel(),
    'Robust':        RobustScaler().fit_transform(data_raw).ravel(),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (title, data) in zip(axes, results_scale.items()):
    ax.boxplot(data)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
plt.suptitle('Выброс 100 влияет на Standard и MinMax; Robust — устойчив', y=1.02)
plt.tight_layout()

### 2.2 Логарифмическое преобразование

Многие реальные распределения имеют **тяжёлый правый хвост** (цены, зарплаты, размер файлов, число запросов). Логарифм приближает их к нормальному — линейные модели от этого выигрывают.

In [ ]:
np.random.seed(42)
data_ln = lognorm(s=1).rvs(1000)

# Тест Шапиро на нормальность (работает на выборке до 5000)
_, p_orig = shapiro(data_ln[:200])
_, p_log  = shapiro(np.log(data_ln[:200]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(data_ln, bins=50, color='orange', edgecolor='white', alpha=0.8)
axes[0].set_title(f'Исходное (лог-нормальное)\np-value Шапиро: {p_orig:.2e}')
axes[1].hist(np.log(data_ln), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title(f'После log-преобразования\np-value Шапиро: {p_log:.3f}')
for ax in axes:
    ax.set_xlabel('Значение')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
print(f"p-value < 0.05 → нормальность отвергается. После log: p={p_log:.3f} — распределение ближе к нормальному.")

### 2.3 Заполнение пропущенных значений

Пропуски — повседневная реальность. Основные стратегии:

| Стратегия | Когда применять |
|---|---|
| Медиана / среднее | Числовые данные, пропуски случайны |
| Мода | Категориальные данные |
| Специальная метка (-999, «Неизвестно») | Факт пропуска несёт информацию |
| Интерполяция | Временные ряды |

> Всегда добавляйте бинарный признак `was_missing` — сам факт пропуска может быть значимым.

In [ ]:
df_miss = pd.DataFrame({
    'age':    [25, np.nan, 35, 40, np.nan, 28],
    'salary': [50000, 60000, np.nan, 80000, 55000, np.nan],
    'city':   ['Москва', 'Казань', None, 'Москва', None, 'Самара'],
})

# Добавляем индикатор пропуска до заполнения
df_miss['age_missing']    = df_miss['age'].isna().astype(int)
df_miss['salary_missing'] = df_miss['salary'].isna().astype(int)

# Заполняем
df_miss['age']    = SimpleImputer(strategy='median').fit_transform(df_miss[['age']]).ravel()
df_miss['salary'] = SimpleImputer(strategy='median').fit_transform(df_miss[['salary']]).ravel()
df_miss['city']   = SimpleImputer(strategy='most_frequent').fit_transform(df_miss[['city']]).ravel()

print(df_miss)

---
## 3. Отбор признаков

Зачем убирать признаки?
1. **Вычислительная сложность**: сотни лишних признаков в продакшне — реальная проблема.
2. **Переобучение**: алгоритм принимает шум за сигнал.

### 3.1 Статистические методы

In [ ]:
np.random.seed(42)
X_gen, y_gen = make_classification(
    n_samples=1000, n_features=20, n_informative=5,
    n_redundant=3, random_state=42
)
print(f"Исходный размер: {X_gen.shape}")
for thr in [0.5, 0.7, 0.9]:
    Xf = VarianceThreshold(thr).fit_transform(X_gen)
    print(f"  VarianceThreshold({thr}): {Xf.shape} — убрано {X_gen.shape[1]-Xf.shape[1]} признаков")

In [ ]:
logit = LogisticRegression(solver='lbfgs', random_state=17, max_iter=1000)

X_kbest = SelectKBest(f_classif, k=5).fit_transform(X_gen, y_gen)
X_varth  = VarianceThreshold(0.9).fit_transform(X_gen)

results = {
    'Все 20 признаков':        cross_val_score(logit, X_gen,   y_gen, cv=5, scoring='neg_log_loss').mean(),
    'SelectKBest (k=5)':       cross_val_score(logit, X_kbest, y_gen, cv=5, scoring='neg_log_loss').mean(),
    'VarianceThreshold (0.9)': cross_val_score(logit, X_varth, y_gen, cv=5, scoring='neg_log_loss').mean(),
}
print("Log-loss (CV), чем ближе к 0 — тем лучше:")
for name, s in results.items():
    print(f"  {name:30s}: {s:.4f}")

### 3.2 Отбор на основе модели

Используем встроенную важность признаков: случайный лес или Lasso отбирают только значимые.

In [ ]:
rf = RandomForestClassifier(n_estimators=50, random_state=17, n_jobs=-1)

pipe_select = make_pipeline(SelectFromModel(estimator=rf), logit)
pipe_lr     = make_pipeline(StandardScaler(), logit)

scores = {
    'LR (все признаки)':         cross_val_score(pipe_lr,     X_gen, y_gen, cv=5, scoring='neg_log_loss').mean(),
    'RF-отбор → LR':             cross_val_score(pipe_select, X_gen, y_gen, cv=5, scoring='neg_log_loss').mean(),
    'RandomForest':              cross_val_score(rf,          X_gen, y_gen, cv=5, scoring='neg_log_loss').mean(),
}
for name, s in scores.items():
    print(f"  {name:35s}: {s:.4f}")

### 3.3 Пошаговый отбор (Sequential Feature Selection)

Добавляем признаки один за другим, каждый раз выбирая тот, что даёт наибольший прирост качества.

In [ ]:
sfs = SequentialFeatureSelector(
    logit, n_features_to_select=5, direction='forward', cv=3, n_jobs=-1
)
sfs.fit(X_gen, y_gen)
selected = np.where(sfs.get_support())[0]
print(f"Отобранные признаки (индексы): {selected}")

X_sfs = sfs.transform(X_gen)
score_sfs = cross_val_score(logit, X_sfs, y_gen, cv=5, scoring='neg_log_loss').mean()
print(f"Log-loss с SFS (5 признаков):  {score_sfs:.4f}")

---
## 4. Ridge и Lasso регрессии

Теперь — регуляризованная регрессия на реальных данных. Мы уже разобрали теорию в Теме 4; здесь наблюдаем поведение на практике.

Датасет: цены на жильё в Калифорнии.

In [ ]:
housing = fetch_california_housing()
X_h = pd.DataFrame(housing.data, columns=housing.feature_names)
y_h = housing.target

print(f"Объектов: {X_h.shape[0]}, признаков: {X_h.shape[1]}")
print(f"Целевая переменная: медианная стоимость дома (в $100k), диапазон [{y_h.min():.2f}, {y_h.max():.2f}]")
print()
print("Описание признаков:")
for name, desc in zip(housing.feature_names, [
    'Медианный доход (в $10k)',
    'Средний возраст домов в квартале',
    'Среднее число комнат на домохозяйство',
    'Среднее число спален на домохозяйство',
    'Население квартала',
    'Среднее число жильцов на домохозяйство',
    'Широта', 'Долгота',
]):
    print(f"  {name:12s}: {desc}")

In [ ]:
# Масштабируем — обязательно перед регуляризацией!
scaler_h = StandardScaler()
X_h_sc = scaler_h.fit_transform(X_h)

### 4.1 Lasso (L1)

$$\mathcal{L}_{\text{lasso}} = \frac{1}{2\ell}\sum_{i=1}^\ell (y_i - \mathbf{w}^T \mathbf{x}_i)^2 + \alpha \|\mathbf{w}\|_1$$

L1 **обнуляет** коэффициенты слабых признаков → встроенный отбор признаков.

In [ ]:
# Путь коэффициентов Lasso
alphas_lasso = np.linspace(0.001, 0.8, 100)
coefs_lasso = []
for a in alphas_lasso:
    m = Lasso(alpha=a, max_iter=5000)
    m.fit(X_h_sc, y_h)
    coefs_lasso.append(m.coef_)

plt.figure(figsize=(10, 5))
plt.plot(alphas_lasso, coefs_lasso)
plt.xlabel('alpha (сила регуляризации)')
plt.ylabel('Значения коэффициентов')
plt.title('Путь коэффициентов Lasso: при большом alpha признаки обнуляются')
plt.legend(housing.feature_names, loc='upper right', fontsize=8, ncol=2)
plt.axhline(0, color='black', linewidth=0.8)
plt.grid(True, alpha=0.3)

In [ ]:
# Подбор оптимального alpha через CV
lasso_cv = LassoCV(alphas=alphas_lasso, cv=5, random_state=17, max_iter=5000)
lasso_cv.fit(X_h_sc, y_h)

print(f"Оптимальный alpha (Lasso): {lasso_cv.alpha_:.4f}")
print(f"Ненулевых признаков: {np.sum(lasso_cv.coef_ != 0)} из {X_h.shape[1]}")
print("\nКоэффициенты:")
for name, coef in zip(housing.feature_names, lasso_cv.coef_):
    marker = '' if coef != 0 else '  <- обнулён'
    print(f"  {name:12s}: {coef:8.4f}{marker}")

mse_lasso = abs(cross_val_score(
    Lasso(lasso_cv.alpha_, max_iter=5000), X_h_sc, y_h,
    cv=5, scoring='neg_mean_squared_error'
).mean())
print(f"\nCV MSE (Lasso): {mse_lasso:.4f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(lasso_cv.alphas_, lasso_cv.mse_path_.mean(1), color='steelblue', lw=2)
plt.axvline(lasso_cv.alpha_, color='orange', linestyle='--',
            label=f'Оптимум: α={lasso_cv.alpha_:.4f}')
plt.xlabel('alpha')
plt.ylabel('MSE (CV)')
plt.title('Lasso: MSE при разных значениях alpha')
plt.legend()
plt.grid(True, alpha=0.3)

### 4.2 Ridge (L2)

$$\mathcal{L}_{\text{ridge}} = \frac{1}{2\ell}\sum_{i=1}^\ell (y_i - \mathbf{w}^T \mathbf{x}_i)^2 + \alpha \|\mathbf{w}\|_2^2$$

L2 **уменьшает** все коэффициенты, но не обнуляет. Особенно хорош при мультиколлинеарности.

In [ ]:
# Путь коэффициентов Ridge
alphas_ridge = np.logspace(-2, 5, 200)
coefs_ridge = []
for a in alphas_ridge:
    m = Ridge(alpha=a)
    m.fit(X_h_sc, y_h)
    coefs_ridge.append(m.coef_)

plt.figure(figsize=(10, 5))
plt.plot(alphas_ridge, coefs_ridge)
plt.xscale('log')
plt.xlabel('alpha (логарифмическая шкала)')
plt.ylabel('Значения коэффициентов')
plt.title('Путь коэффициентов Ridge: все коэффициенты уменьшаются, но не обнуляются')
plt.legend(housing.feature_names, loc='upper right', fontsize=8, ncol=2)
plt.axhline(0, color='black', linewidth=0.8)
plt.grid(True, alpha=0.3)

In [ ]:
ridge_cv = RidgeCV(alphas=alphas_ridge, scoring='neg_mean_squared_error', cv=5)
ridge_cv.fit(X_h_sc, y_h)

mse_ridge = abs(cross_val_score(
    Ridge(ridge_cv.alpha_), X_h_sc, y_h, cv=5, scoring='neg_mean_squared_error'
).mean())
mse_lr = abs(cross_val_score(
    LinearRegression(), X_h_sc, y_h, cv=5, scoring='neg_mean_squared_error'
).mean())

print(f"Оптимальный alpha (Ridge): {ridge_cv.alpha_:.2f}")
print()
print("Итоговое сравнение (CV MSE, меньше = лучше):")
for name, mse in [
    ('Линейная регрессия (без регул.)', mse_lr),
    ('Ridge (L2)',                     mse_ridge),
    ('Lasso (L1)',                     mse_lasso),
]:
    print(f"  {name:35s}: {mse:.4f}")

### 4.3 Сравнение коэффициентов Lasso и Ridge

In [ ]:
coef_df = pd.DataFrame({
    'Ridge': ridge_cv.coef_,
    'Lasso': lasso_cv.coef_,
}, index=housing.feature_names)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, color, alpha_val in zip(
    axes,
    ['Ridge', 'Lasso'],
    ['steelblue', 'orange'],
    [ridge_cv.alpha_, lasso_cv.alpha_]
):
    coef_df[col].plot(kind='barh', ax=ax, color=color, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{col} (alpha={alpha_val:.4f})')
    ax.grid(True, alpha=0.3, axis='x')
plt.suptitle('Сравнение коэффициентов: Ridge не обнуляет, Lasso — обнуляет слабые', y=1.02)
plt.tight_layout()

| | Lasso (L1) | Ridge (L2) |
|---|---|---|
| **Эффект на коэффициенты** | Обнуляет слабые | Уменьшает все |
| **Отбор признаков** | Встроенный | Нет |
| **Мультиколлинеарность** | Выбирает один из группы | Распределяет вес по группе |
| **Когда использовать** | Много нерелевантных признаков | Все признаки потенциально важны |

---
## 5. Практика

Датасет: цены на жильё в Калифорнии (`X_h`, `y_h`, `X_h_sc` уже загружены выше).

In [ ]:
print("Признаки:", list(X_h.columns))
print(f"Объектов: {X_h.shape[0]}")
X_h.describe().round(2)

### Задание 1
Создайте расширенный датафрейм `X_eng` с тремя дополнительными признаками:
- `rooms_per_household` = `AveRooms` / `AveOccup`
- `bedrooms_ratio` = `AveBedrms` / `AveRooms`
- `pop_per_household` = `Population` / `HouseAge`

Обучите `RidgeCV` на исходных признаках и на `X_eng`. Стал ли CV MSE лучше?

> Не забудьте `StandardScaler` перед регрессией.

In [ ]:
# Ваш код здесь

### Задание 2
Постройте «путь регуляризации» Lasso для `X_eng` (после масштабирования):
- Переберите `alpha` от 0.001 до 1.0 (100 значений)
- Постройте график коэффициентов vs `alpha`
- Какой признак «обнуляется» последним?

In [ ]:
# Ваш код здесь

### Задание 3
Примените гармоническую кодировку к признаку `HouseAge`, предположив цикличность с периодом 52 года (примерный диапазон в датасете):
- Замените `HouseAge` на `HouseAge_cos` и `HouseAge_sin`
- Сравните CV MSE Ridge до и после замены

Ответьте: стало ли лучше? Почему или почему нет?

In [ ]:
# Ваш код здесь

### Задание 4
Постройте пайплайн: `StandardScaler` → `SelectFromModel(Lasso(alpha=0.01))` → `Ridge` с `RidgeCV`.

Сколько признаков отобрал Lasso? Сравните CV MSE с простой Ridge без отбора.

In [ ]:
# Ваш код здесь

### Задание 5 (повышенная сложность)
Полный пайплайн инженерии признаков:
1. Создайте не менее 5 новых признаков (взаимодействия, отношения, логарифмы)
2. Отберите лучшие 6 с помощью `SelectKBest(f_regression, k=6)`
3. Обучите `LassoCV` и `RidgeCV` на отобранных признаках
4. Сравните с базовой `LinearRegression` без инженерии

Выведите итоговую таблицу CV MSE.

In [ ]:
# Ваш код здесь

---
## Полезные ресурсы

- [sklearn: Preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html)
- [sklearn: Feature Selection](https://scikit-learn.org/stable/modules/feature_selection.html)
- [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) / [LassoCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html)
- [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) / [RidgeCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html)
- [SequentialFeatureSelector](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html)
